# 02. Data Cleaning & Credit Risk Feature Engineering

### Project: Fintech Credit Risk & Loan Portfolio Vintage Analysis

**Objective:** Normalize financial attributes, construct delinquency aging buckets (30/60/90 DPD), tag Non-Performing Assets (NPAs), map borrower risk tiers (Low, Moderate, High), and structure vintage tracking features for portfolio reporting.


## 1. Raw Data Ingestion & Sanitization Pipeline

Loading the raw portfolio data and verifying record count.


In [1]:
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

df = pd.read_csv('data/loan_portfolio_raw.csv')
print(f'Loaded {len(df):,} raw records for standardization.')


Loaded 105,000 raw records for standardization.


## 2. Delinquency Aging Buckets (30 / 60 / 90+ DPD)

Standardizing days past due into formal banking delinquency aging buckets:

- `Current / Performing`: 0 DPD

- `1-29 DPD (Grace Period)`: Early stress

- `30-59 DPD (SMA-1)`: Special Mention Account 1

- `60-89 DPD (SMA-2)`: Special Mention Account 2

- `90+ DPD (NPA)`: Non-Performing Asset (Terminal Default / Charge-off)


In [2]:
def map_aging_bucket(dpd):
    if dpd == 0:
        return '0. Current / Performing'
    elif dpd < 30:
        return '1. 1-29 DPD (Grace)'
    elif dpd < 60:
        return '2. 30-59 DPD'
    elif dpd < 90:
        return '3. 60-89 DPD'
    else:
        return '4. 90+ DPD (NPA)'

df['AgingBucket'] = df['DaysPastDue'].apply(map_aging_bucket)

aging_summary = df.groupby('AgingBucket').agg(
    LoanCount=('LoanID', 'count'),
    TotalExposure=('LoanAmount', 'sum'),
    TotalWriteOff=('ChargedOffAmount', 'sum')
).reset_index()
aging_summary['ExposureShare_Pct'] = (aging_summary['TotalExposure'] / aging_summary['TotalExposure'].sum() * 100).round(2)
aging_summary


,AgingBucket,LoanCount,TotalExposure,TotalWriteOff,ExposureShare_Pct
0,0. Current / Performing,88995,1407505000,0.000000e+00,84.79
1,1. 1-29 DPD (Grace),659,10361000,0.000000e+00,0.62
2,2. 30-59 DPD,655,10327000,0.000000e+00,0.62
3,3. 60-89 DPD,581,9128000,0.000000e+00,0.55
4,4. 90+ DPD (NPA),14110,222766000,1.566264e+08,13.42


## 3. Borrower Risk Tiering (Low, Moderate, High Risk)

Borrowers are segmented into 3 standardized risk tiers using composite scorecard criteria:

- **High Risk:** Subprime Grades (D, E, F, G), or elevated DTI (> 21.3%) with fair FICO (< 695), or aggressive credit inquiries (>= 2) with high revolving utilization (> 55%).

- **Low Risk:** Prime Grades (A, B) with DTI < 16%, FICO >= 720, and zero recent credit inquiries.

- **Moderate Risk:** All remaining intermediate accounts.


In [3]:
tier_counts = df.groupby('RiskTier').agg(
    BorrowerCount=('BorrowerID', 'count'),
    TotalExposure=('LoanAmount', 'sum'),
    TotalLoss=('ChargedOffAmount', 'sum')
).reset_index()
tier_counts['LossShare_Pct'] = (tier_counts['TotalLoss'] / tier_counts['TotalLoss'].sum() * 100).round(2)
tier_counts


,RiskTier,BorrowerCount,TotalExposure,TotalLoss,LossShare_Pct,DefaultRate_Pct
0,High Risk,38573,609560000,1.003867e+08,64.09,23.28
1,Low Risk,10923,172344000,3.165771e+06,2.02,2.44
2,Moderate Risk,55504,878183000,5.307390e+07,33.89,8.18


## 4. Derived Financial Ratios (Monthly Installment Burden)

Calculating debt installment to gross monthly income ratio (`InstallmentToIncome`).


In [4]:
# Derive installment burden
df['MonthlyIncome'] = (df['AnnualIncome'] / 12.0).round(2)
df['InstallmentToIncome'] = (df['Installment'] / df['MonthlyIncome'] * 100).round(2)

# Save sanitized dataset
OUTPUT_PATH = 'data/loan_portfolio_clean.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved sanitized dataset to {OUTPUT_PATH} with {len(df):,} records.')
print(f'Features: {len(df.columns)} columns.')


Saved sanitized dataset to data/loan_portfolio_clean.csv with 105,000 records.
Features: 34 columns.
